# Google Play Store Category Insights
Load the Google Play Store dataset, summarize categories, and visualize supply vs demand.


In [ ]:
from pathlib import Path

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_rows", 200)


In [ ]:
DATASET_SLUG = "lavanya/google-play-store-apps"
LOCAL_CANDIDATES = [Path("googleplaystore.csv"), Path("data/googleplaystore.csv")]
REQUIRED_COLUMNS = {"Category", "Installs", "Rating", "Reviews"}


def load_play_store_data(dataset_slug=DATASET_SLUG):
    for candidate in LOCAL_CANDIDATES:
        if candidate.exists():
            return pd.read_csv(candidate)

    try:
        dataset_path = Path(kagglehub.dataset_download(dataset_slug))
    except Exception as exc:
        raise RuntimeError(
            "Dataset download failed. Update DATASET_SLUG or place googleplaystore.csv in the project root or data/."
        ) from exc

    csv_paths = list(dataset_path.rglob("*.csv"))
    for csv_path in csv_paths:
        columns = pd.read_csv(csv_path, nrows=0).columns
        if REQUIRED_COLUMNS.issubset(columns):
            return pd.read_csv(csv_path)

    raise FileNotFoundError(
        "Could not locate a CSV with Category, Installs, Rating, and Reviews columns."
    )


df = load_play_store_data()
df.head()


In [ ]:
df = df.copy()

df["Category"] = df["Category"].astype(str).str.strip()
df = df[df["Category"].str.match(r"^[A-Za-z_]+$")]

df["Installs"] = (
    df["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)
df["Installs"] = pd.to_numeric(df["Installs"], errors="coerce").fillna(0)

df["Reviews"] = pd.to_numeric(df["Reviews"], errors="coerce").fillna(0)

df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

app_column = "App" if "App" in df.columns else "Category"

summary = (
    df.groupby("Category", dropna=False)
    .agg(
        **{
            "Number of Apps": (app_column, "count"),
            "Total Installs": ("Installs", "sum"),
            "Average Rating": ("Rating", "mean"),
            "Total Reviews": ("Reviews", "sum"),
        }
    )
    .reset_index()
)

summary["Number of Apps"] = summary["Number of Apps"].astype(int)
summary["Total Installs"] = summary["Total Installs"].round(0).astype(int)
summary["Total Reviews"] = summary["Total Reviews"].round(0).astype(int)
summary["Average Rating"] = summary["Average Rating"].round(2)

summary_sorted = summary.sort_values(
    by=["Total Installs", "Number of Apps"], ascending=[False, True]
).reset_index(drop=True)

summary_sorted.head()


In [ ]:
summary_sorted


In [ ]:
apps_sorted = summary_sorted.sort_values("Number of Apps", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=apps_sorted, x="Category", y="Number of Apps", color="#4c72b0")
plt.title("Number of Apps by Category")
plt.xlabel("Category")
plt.ylabel("Number of Apps")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
installs_sorted = summary_sorted.sort_values("Total Installs", ascending=False)

plt.figure(figsize=(12, 6))
sns.barplot(data=installs_sorted, x="Category", y="Total Installs", color="#55a868")
plt.title("Total Installs by Category")
plt.xlabel("Category")
plt.ylabel("Total Installs")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## Category insights
Heuristic signals derived from installs per app, app counts, and ratings.


In [ ]:
from IPython.display import Markdown, display

summary_insights = summary_sorted.copy()
summary_insights["Average Rating"] = summary_insights["Average Rating"].fillna(
    summary_insights["Average Rating"].mean()
)
summary_insights["Installs per App"] = (
    summary_insights["Total Installs"] / summary_insights["Number of Apps"]
)

apps_q75 = summary_insights["Number of Apps"].quantile(0.75)
apps_q25 = summary_insights["Number of Apps"].quantile(0.25)
installs_median = summary_insights["Installs per App"].median()
overall_rating = summary_insights["Average Rating"].mean()

overcrowded = (
    summary_insights[summary_insights["Number of Apps"] >= apps_q75]
    .sort_values("Installs per App")
    .head(2)
)
underserved = (
    summary_insights[summary_insights["Number of Apps"] <= apps_q25]
    .sort_values("Installs per App", ascending=False)
    .head(2)
)
high_demand_low_quality = (
    summary_insights[
        (summary_insights["Installs per App"] >= installs_median)
        & (summary_insights["Average Rating"] < overall_rating)
    ]
    .sort_values("Average Rating")
    .head(1)
)

insights = []
for _, row in overcrowded.iterrows():
    insights.append(
        "Overcrowded: {category} has {apps:,} apps but only {ipa:,.0f} installs per app.".format(
            category=row["Category"],
            apps=int(row["Number of Apps"]),
            ipa=row["Installs per App"],
        )
    )

for _, row in underserved.iterrows():
    insights.append(
        "Underserved: {category} has just {apps:,} apps yet {ipa:,.0f} installs per app.".format(
            category=row["Category"],
            apps=int(row["Number of Apps"]),
            ipa=row["Installs per App"],
        )
    )

for _, row in high_demand_low_quality.iterrows():
    insights.append(
        "High-demand but low-quality: {category} shows {ipa:,.0f} installs per app with an average rating of {rating:.2f}.".format(
            category=row["Category"],
            ipa=row["Installs per App"],
            rating=row["Average Rating"],
        )
    )

insights = insights[:5]

if len(insights) < 5:
    insights.extend(["No idea generated - retry later."] * (5 - len(insights)))

insight_markdown = "\n".join(f"- {insight}" for insight in insights)
display(Markdown(insight_markdown))
